# Lab 3 — European Rain Forecast: a Minimal Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-agent.ipynb)

The smallest agent worth submitting to challenge 177:
for each city and calendar month, the 100 quantiles of hourly rain
over 2020–2024, used as 100 members. No input is read, and it still
beats the challenge's reference — the analysis notebook showed why.

You will build the table, write `agent.py`, call it on the exact
request the challenge sends, score it the way the challenge scores
it over 14 months it has not seen, and submit it.

**Time:** 15 minutes. **Deliverable:** your agent on the leaderboard
of challenge 177, and its local score from section 8.

---

## 1. The contract

Every 6 hours the challenge calls your agent **once**, with the whole
panel: `Agent().predict(request) -> dict`.

| request key | shape | what it is |
|---|---|---|
| `issue_time` | str | ISO UTC hour the forecast is made from |
| `timestamps` | 48 str | the input hours, ascending, ending at `issue_time` |
| `observed` | 48 bool | `False`: the feed dropped that hour, the previous one was carried forward |
| `cities` | 45 dict | `name`, `country`, `lat`, `lon`, in a fixed order |
| `feature_names` | 8 str | `temperature`, `rain`, `wind_speed`, `wind_direction`, `humidity`, `clouds`, `visibility`, `snow` |
| `history` | (45, 48, 8) | cities × hours × features |
| `horizons` | [6, 48] | lead times, hours after `issue_time` |
| `max_members` | 100 | the largest M allowed |

The response is `{"rain": (45, 2, M)}` as nested lists: for each city
and horizon, M plausible amounts of rain in mm at `issue_time + h`.
1 ≤ M ≤ 100, the same M everywhere, every value finite; values below
0 are clipped to 0. Samples or quantiles, both are members. M = 1 is
a deterministic forecast.

Each forecast is scored **once**, 48 hours later, when both hours have
been observed: the CRPS of your members against the rain that fell,
compared with the **reference** — the city's own last 48 hours as 48
members — per horizon, and divided by a fixed constant C_h for the
month. Section 4 writes it out.

The call has 60 s, 3 CPUs and 3 GiB. Files you upload sit next to
`agent.py`.

---

## 2. Setup and download

The notebook needs one secret, `MLARENA_API_KEY`: ML-Arena, Profile →
API Keys (it starts with `mlk_user_`). Never paste its value into a
cell: a notebook is shared with its code and its outputs.

In Colab, add it to the *Secrets* panel (the key icon on the left) and
allow this notebook to access it. Outside Colab, set it in your
environment before starting Jupyter; the cell raises if it is missing.
Colab already has pandas, numpy and matplotlib; the only install is
the ML-Arena client.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlarena-sdk"], check=True)
    from google.colab import userdata
    os.environ["MLARENA_API_KEY"] = userdata.get("MLARENA_API_KEY")

import matplotlib.pyplot as plt
import mlarena
import numpy as np
import pandas as pd
import requests

pd.set_option("display.width", 120)
client = mlarena.connect(api_key=os.environ["MLARENA_API_KEY"])
CHALLENGE_ID = 177

Challenge 177 carries two datasets: the 28 MB European panel this lab
uses, and seven yearly files for ~990 cities worldwide (4.4 GB).
`client.download_dataset(177)` would fetch all of them, so the cell
below asks for the list, picks the one file by its label, and fetches
its signed `download_url` (valid one hour, no key needed).

In [ ]:
FILE = "weather_europe_2020_2026.csv.gz"

if not os.path.exists(FILE):
    listing = client.datasets(CHALLENGE_ID)
    [meta] = [f for ds in listing["datasets"] for f in ds["files"]
              if f["label"] == FILE]
    resp = requests.get(meta["download_url"], timeout=300)
    resp.raise_for_status()
    with open(FILE, "wb") as fh:
        fh.write(resp.content)
print(FILE, f"{os.path.getsize(FILE) / 1e6:.1f} MB")

---

## 3. The data, as the challenge sees it

The challenge sends the cities in its own fixed order, not the
file's alphabetical one. `X` is the whole file as an array (hours,
cities, features) in that order, so a request is a slice of it.

In [ ]:
PANEL = [
    ("Amsterdam", "NL"), ("Athens", "GR"), ("Belgrade", "RS"),
    ("Berlin", "DE"), ("Brussels", "BE"), ("Bucharest", "RO"),
    ("Budapest", "HU"), ("Chisinau", "MD"), ("Copenhagen", "DK"),
    ("Dublin", "IE"), ("Helsinki", "FI"), ("Kyiv", "UA"),
    ("London", "GB"), ("Madrid", "ES"), ("Minsk", "BY"),
    ("Moscow", "RU"), ("Oslo", "NO"), ("Paris", "FR"),
    ("Prague", "CZ"), ("Riga", "LV"), ("Rome", "IT"),
    ("Sarajevo", "BA"), ("Sofia", "BG"), ("Stockholm", "SE"),
    ("Vienna", "AT"), ("Warsaw", "PL"), ("Zagreb", "HR"),
    ("Istanbul", "TR"), ("Saint Petersburg", "RU"), ("Hamburg", "DE"),
    ("Munich", "DE"), ("Frankfurt am Main", "DE"), ("Milan", "IT"),
    ("Naples", "IT"), ("Palermo", "IT"), ("Barcelona", "ES"),
    ("Valencia", "ES"), ("Sevilla", "ES"), ("Marseille", "FR"),
    ("Birmingham", "GB"), ("Glasgow", "GB"), ("Kraków", "PL"),
    ("Göteborg", "SE"), ("Odesa", "UA"), ("Kharkiv", "UA"),
]
FEATURES = ["temperature", "rain", "wind_speed", "wind_direction",
            "humidity", "clouds", "visibility", "snow"]

df = pd.read_csv(FILE, dtype={"city_name": "category",
                              "country_code": "category"})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True,
                                 format="ISO8601")
key = df["city_name"].astype(str) + "|" + df["country_code"].astype(str)
position = {f"{n}|{c}": j for j, (n, c) in enumerate(PANEL)}
df["pos"] = key.map(position)
assert df["pos"].notna().all() and df["pos"].nunique() == len(PANEL)
df = df.sort_values(["timestamp", "pos"])

hours = pd.DatetimeIndex(df["timestamp"].unique())
assert (hours[1:] - hours[:-1] == pd.Timedelta(hours=1)).all()
X = df[FEATURES].to_numpy(float).reshape(len(hours), len(PANEL),
                                         len(FEATURES))
coords = df.groupby("pos")[["latitude", "longitude"]].first()
rain = X[:, :, FEATURES.index("rain")]          # (hours, cities)
month = hours.month.to_numpy()
del df, key
print(X.shape, hours[0], "->", hours[-1])

The protocol, fixed before anything is fitted: the model learns from
**2020–2024**; it is scored on the forecasts issued from **2025-01-01**
on, every 6 hours at the UTC hours the challenge uses, up to the last
one whose +48 h is in the file.

In [ ]:
HORIZONS = [6, 48]
ISSUE_HOURS = (5, 11, 17, 23)          # UTC, every 6 hours
SPLIT = hours.searchsorted(pd.Timestamp("2025-01-01", tz="UTC"))

issues = np.arange(SPLIT, len(hours) - max(HORIZONS))
issues = issues[np.isin(hours.hour[issues], ISSUE_HOURS)]
print(len(issues), "forecasts,", hours[issues[0]], "->",
      hours[issues[-1]])

---

## 4. The scorer

The CRPS of M members x₁ … x_M against the observed rain y:

$$\mathrm{CRPS} = \frac{1}{M}\sum_i |x_i - y|
- \frac{1}{2M^2}\sum_i\sum_j |x_i - x_j|$$

For each run and horizon, with the means over the 45 cities, and
C_h[month] = `REFERENCE_CRPS[h][month]`, the reference's mean CRPS in
the calendar month of the valid hour over 2020–2024 (a fixed table
the challenge publishes):

$$\mathrm{skill}_h = \max\left(-1,\ 
\frac{\overline{\mathrm{CRPS}}_{ref}
- \overline{\mathrm{CRPS}}_{agent}}{C_h[\mathrm{month}]}\right)$$

The run scores the mean of skill₆ and skill₄₈; the leaderboard is the
mean over runs. 0 is the reference, positive is better, and a
perfect forecast scores the reference's CRPS over C_h — about 1 on
average, more in a wet run. The denominator does not depend on the
weather, so the score rewards exactly what the CRPS rewards.

In [ ]:
# C_h: the reference's mean CRPS (mm) per month of the valid hour,
# Jan..Dec, over 2020-2024 -- the challenge's published table.
REFERENCE_CRPS = {
    6: np.array([0.059, 0.063, 0.062, 0.069, 0.088, 0.088,
                 0.085, 0.084, 0.093, 0.098, 0.087, 0.077]),
    48: np.array([0.066, 0.069, 0.067, 0.075, 0.094, 0.094,
                  0.089, 0.090, 0.099, 0.105, 0.091, 0.082]),
}


def crps(members, y):
    """CRPS of ensembles (..., M) against outcomes (...), in mm."""
    x = np.sort(np.maximum(members, 0.0), axis=-1)  # clipped at 0
    M = x.shape[-1]
    k = np.arange(1, M + 1)
    spread = (x * (2 * k - M - 1)).sum(axis=-1) / M ** 2
    return np.abs(x - y[..., None]).mean(axis=-1) - spread


def skill(members, reference, y, h, valid_month):
    """skill_h of runs, and the two mean CRPS it compares.

    members (..., cities, M), reference (..., cities, 48),
    y (..., cities), valid_month (...) in 1..12.
    """
    agent = crps(members, y).mean(axis=-1)
    ref = crps(reference, y).mean(axis=-1)
    c_h = REFERENCE_CRPS[h][valid_month - 1]
    s = np.maximum((ref - agent) / c_h, -1.0)
    return s, agent, ref

---

## 5. The model: quantiles per city and month

For each city and calendar month, the quantiles of hourly rain at the
levels 0.005, 0.015, …, 0.995. About 85% of hours are dry, so most
members are 0 and the top ones spread over the wet-hour amounts: a
zero-inflated distribution, read straight off the data.

In [ ]:
M = 100
LEVELS = (np.arange(M) + 0.5) / M


def fit_table(end):
    """(12, cities, M) quantiles of the rain of hours [0, end)."""
    r, m = rain[:end], month[:end]
    return np.stack([np.quantile(r[m == k], LEVELS, axis=0).T
                     for k in range(1, 13)])


table = fit_table(SPLIT)
glasgow = table[:, [n for n, _ in PANEL].index("Glasgow")]
print("Glasgow, January: zero members", (glasgow[0] == 0).sum(),
      "| top five", glasgow[0, -5:].round(2))
print("Glasgow, July:    zero members", (glasgow[6] == 0).sum(),
      "| top five", glasgow[6, -5:].round(2))

Would the hour of day help? The analysis notebook found a strong
afternoon peak in summer. Split each cell by the UTC hour of the valid
hour, and score both tables over the held-out forecasts — vectorised,
with the scorer above:

In [ ]:
utc = hours.hour.to_numpy()
by_hour = np.stack([
    np.stack([np.quantile(rain[:SPLIT][(month[:SPLIT] == k)
                                      & (utc[:SPLIT] == u)],
                          LEVELS, axis=0).T for u in range(24)])
    for k in range(1, 13)])                 # (12, 24, cities, M)

reference = np.stack([rain[i - 47:i + 1].T for i in issues])
rows = {}
for h in HORIZONS:
    y, vm, vh = rain[issues + h], month[issues + h], utc[issues + h]
    for name, members in (("city x month", table[vm - 1]),
                          ("city x month x hour",
                           by_hour[vm - 1, vh])):
        s, a, _ = skill(members, reference, y, h, vm)
        rows.setdefault(name, {})[f"skill {h}h"] = s.mean()
        rows[name][f"CRPS {h}h"] = a.mean()
print(pd.DataFrame(rows).T.round(4))
del by_hour

**Finding.** The hour loses at both horizons: skill 0.0239 against
0.0264 at +6 h, 0.0862 against 0.0885 at +48 h. A city × month cell
holds about 3,600 hours; split by hour, about 150, and 100 quantiles
of 150 mostly-zero values are noisier than the diurnal signal is
worth. The agent stays at city × month. Save its table:

In [ ]:
import json

TABLE_FILE = "rain_quantiles.json"


def save_table(table, path=TABLE_FILE):
    """{"Name|CC": 12 months x M members}, rounded to 0.01 mm."""
    quantiles = {f"{n}|{c}": np.round(table[:, j], 2).tolist()
                 for j, (n, c) in enumerate(PANEL)}
    with open(path, "w") as f:
        json.dump({"members": M, "quantiles": quantiles}, f)
    print(path, f"{os.path.getsize(path) / 1e3:.0f} kB")


save_table(table)

---

## 6. `agent.py`

The challenge imports `agent.py`, builds `Agent()` once with no
argument, and calls `predict`. The table is read from the directory
of `agent.py` itself, where the uploaded files are, never from the
working directory.

It needs nothing but the standard library, so it runs on the lightest
runtime, 181 (Python 3.12 with numpy and pandas). The cities are
looked up by name, not by position, so a change of order in the
request cannot silently mix them up.

In [ ]:
%%writefile agent.py
"""European Rain Forecast: city x month climatology, 100 members."""
import json
import os
from datetime import datetime, timedelta

HERE = os.path.dirname(os.path.abspath(__file__))


class Agent:
    def __init__(self):
        with open(os.path.join(HERE, "rain_quantiles.json")) as f:
            self.quantiles = json.load(f)["quantiles"]

    def predict(self, request):
        issue = datetime.fromisoformat(
            request["issue_time"].replace("Z", "+00:00"))
        months = [(issue + timedelta(hours=h)).month
                  for h in request["horizons"]]
        rain = []
        for city in request["cities"]:
            by_month = self.quantiles[
                f"{city['name']}|{city['country']}"]
            rain.append([by_month[m - 1] for m in months])
        return {"rain": rain}

---

## 7. A real request

`make_request(i)` builds what the challenge sends when hour `i` is
the issue time: the 48 hours ending at it, the 45 cities in panel
order, the 8 features. The feed never reports `visibility`, and the
challenge sends it as 0. Every hour of the file was observed, so
`observed` is all `True` here; live, a dropped hour is `False` and
holds the previous hour's values.

In [ ]:
import importlib
import time

import agent

importlib.reload(agent)             # pick up any edit to agent.py


def iso(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def make_request(i):
    window = X[i - 47:i + 1].transpose(1, 0, 2)   # (cities, 48, 8)
    return {
        "issue_time": iso(hours[i]),
        "timestamps": [iso(t) for t in hours[i - 47:i + 1]],
        "observed": [True] * 48,
        "cities": [{"name": n, "country": c,
                    "lat": float(coords.loc[j, "latitude"]),
                    "lon": float(coords.loc[j, "longitude"])}
                   for j, (n, c) in enumerate(PANEL)],
        "feature_names": FEATURES,
        "history": np.nan_to_num(window, nan=0.0).tolist(),
        "horizons": HORIZONS,
        "max_members": 100,
    }


def check(response, request):
    """The response as a (cities, horizons, M) array, or raise."""
    json.dumps(response, allow_nan=False)      # JSON, no NaN/inf
    ens = np.asarray(response["rain"], dtype=float)
    shape = (len(request["cities"]), len(request["horizons"]))
    assert ens.ndim == 3 and ens.shape[:2] == shape, ens.shape
    assert 1 <= ens.shape[2] <= request["max_members"], ens.shape
    assert np.isfinite(ens).all()
    return ens


request = make_request(issues[0])
json.dumps(request, allow_nan=False)

bot = agent.Agent()
t0 = time.perf_counter()
ens = check(bot.predict(request), request)
print(request["issue_time"], "->", ens.shape,
      f"in {1e3 * (time.perf_counter() - t0):.1f} ms")

One call for the whole panel, in milliseconds — the budget is 60 s.

---

## 8. The local score

Replay the challenge over every held-out issue time: build the
request, call the agent, score both horizons against the rain that
fell, with the reference built from the same request.

In [ ]:
RAIN = FEATURES.index("rain")
rows = []
for i in issues:
    request = make_request(i)
    ens = check(bot.predict(request), request)
    reference = np.asarray(request["history"])[:, :, RAIN]
    row = {"issue": hours[i]}
    for k, h in enumerate(HORIZONS):
        s, a, r = skill(ens[:, k], reference, rain[i + h], h,
                        month[i + h])
        row |= {f"skill {h}h": s, f"CRPS {h}h": a,
                f"reference {h}h": r}
    rows.append(row)

runs = pd.DataFrame(rows).set_index("issue")
runs["score"] = runs[["skill 6h", "skill 48h"]].mean(axis=1)
se = runs["score"].std() / np.sqrt(len(runs))
print(runs.mean().round(4).to_string())
print(f"\nlocal score {runs['score'].mean():+.4f} ± {se:.4f}"
      f" over {len(runs)} runs")

**Finding.** +0.0574 over 1,700 runs: +0.026 at +6 h, +0.089 at +48 h
— the climatology row of the analysis notebook, reproduced through
the real request, the real `agent.py` and the JSON file. Its CRPS is
0.0760 mm against the reference's 0.0781 at +6 h, and 0.0758 against
0.0830 at +48 h: most of its lead is at +48 h, where the last 48
hours of one city are a poor sample of its climate.

The ± treats the runs as independent. Consecutive runs share the
same weather: computed from weekly blocks, the standard error is
about twice as large, 0.005. Compare two agents run by run, on the
same runs, before believing a gap of a few thousandths.

---

## 9. Submit

The held-out score was measured once; now refit the table on every
hour in the file, so the submitted agent knows the most recent
climate too, and check the refitted agent once more.

In [ ]:
save_table(fit_table(len(hours)))
importlib.reload(agent)
request = make_request(len(hours) - 1)
print(check(agent.Agent().predict(request), request).shape)

`client.submit` creates the submission, uploads the two files, pins
the runtime and deploys it. Without `runtime_id` the challenge's
default runtime is used; 181 is the plain Python one, enough for an
agent that imports nothing but the standard library (182 adds
scikit-learn, LightGBM and XGBoost).

**Run this cell once.** Each run creates a new submission.

In [ ]:
result = client.submit(CHALLENGE_ID,
                       files=["agent.py", TABLE_FILE],
                       submission_name="city-month-climatology",
                       runtime_id=181)
submission_id = result["submission_id"]
print(submission_id)

`status` goes `deploy_queue` → `deploy_run` → `active`; a failed
deploy says why in `last_status_message`. Once `active`, the agent
is called at every run, every 6 hours.

In [ ]:
status = client.status(submission_id, CHALLENGE_ID)
print(status["status"], "|", status["last_status_message"])
for run in status["run_info"]["results"][-5:]:
    print(run["created_at_ts"], run["job_status"],
          run["submission_reward"])

**The first score appears about 48 hours after the first forecast.**
A forecast is scored once both of its hours, +6 h and +48 h, have
happened, so the first eight runs of a new agent report no score
(`None`), not 0. From then on every run scores the forecast made 48
hours earlier. The leaderboard's `MeanReward` is the mean over the
scored runs.

In [ ]:
board = client.leaderboard(CHALLENGE_ID)
if board.empty:
    print("no scored submission yet")
else:
    print(board[["Rank", "SubmissionName", "Username", "MeanReward",
                 "NumberOfRuns"]].head(10).to_string(index=False))

---

## 10. Where to go next

The climatology reads no input. The challenge's own **starter
agent** (the template on the challenge page) does the opposite: it
reads only the 48-hour window. Its chance of rain is the city's wet
share over those 48 hours, shrunk halfway toward the panel's, raised
by half if it is raining now and lowered by a fifth if not (+6 h
only); its amounts are the wet hours seen anywhere in the panel. It
adapts to the weather of the week and knows nothing of the season.

| agent | score | skill +6 h | skill +48 h |
|---|---|---|---|
| this climatology | +0.057 | +0.026 | +0.089 |
| the starter agent | +0.076 | +0.070 | +0.081 |
| +6 h split by rain in the last 3 h, +48 h this climatology | +0.080 | +0.073 | +0.088 |

The first row is section 8; the other two are the challenge's own
replay, over the same 1,700 runs. The starter wins at +6 h, where
the current state matters; the
climatology wins at +48 h, where it does not (section 6 of the
analysis). The last row takes each where it wins. The next steps:

- **A hurdle model on runtime 182.** A calibrated LightGBM
  classifier for P(wet) and LightGBM quantile regressors
  (`objective="quantile"`)
  for the amount given wet, one per level; the members are the
  mixture's quantiles. Or one quantile LightGBM per level on all
  hours, sorted to repair crossings. Save the boosters next to
  `agent.py` (`booster.save_model(...)`), load them from `HERE`, and
  submit with `runtime_id=182`.
- **Features from the analysis.** Rain over the last 1, 6 and 48
  hours, cloud cover and humidity at the issue time, the rain in the
  city upwind, the solar hour and the month, the city itself. One
  row per city and issue time, one target per horizon.
- **Validation like the challenge.** Train on the past, validate on
  a later block, with a 48-hour gap so no training target falls in
  the validation window; score with `skill` above, averaged over
  runs. Never a random split: neighbouring hours are near copies.
- **Optuna, by the rules.** A fixed `n_trials`, `learning_rate` and
  `reg_lambda` on a log scale, a seeded `TPESampler`, the objective
  returning the validation skill — then the held-out 2025–2026
  score computed once.

The lessons *Probabilistic Prediction* (hurdle models, quantiles as
members, the CRPS) and *Hyperparameter Optimisation* (budget, log
scales, seeds, the multiple-comparisons trap) have the recipes.